## 1. Importar Librerías Requeridas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import time
import threading
import random
import os
from glob import glob
from collections import defaultdict
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Librerías específicas para modelos
from ultralytics import YOLO
import onnxruntime as ort
from text_converter import TextConverter

# Librería para monitoreo de GPU
try:
    import pynvml
    pynvml.nvmlInit()
    GPU_AVAILABLE = True
    print("✅ NVIDIA GPU detectada y pynvml inicializado")
    
    # Información de la GPU
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    gpu_name = pynvml.nvmlDeviceGetName(handle)
    memory_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 Memoria Total: {memory_info.total / 1024**3:.2f} GB")
    
except Exception as e:
    GPU_AVAILABLE = False
    print(f"⚠️  No se pudo inicializar pynvml: {e}")
    print("   Instalar con: pip install pynvml")

# Configuración para reproducibilidad
np.random.seed(42)
random.seed(42)

# Configuración de matplotlib
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print(ort.get_available_providers())

print("\n✅ Librerías importadas correctamente")

## 2. Detectar Proveedores de Ejecución Disponibles

In [ ]:
# Detectar proveedores de ejecución ONNX disponibles
available_providers = ort.get_available_providers()

print("🔍 Proveedores ONNX Runtime disponibles:")
for provider in available_providers:
    print(f"   • {provider}")

# Seleccionar proveedor de ejecución
if 'CUDAExecutionProvider' in available_providers:
    EXECUTION_PROVIDER = 'CUDAExecutionProvider'
    print(f"\n✅ Usando: {EXECUTION_PROVIDER}")
elif 'DmlExecutionProvider' in available_providers:
    EXECUTION_PROVIDER = 'DmlExecutionProvider'
    print(f"\n✅ Usando: {EXECUTION_PROVIDER} (DirectML)")
elif 'TensorrtExecutionProvider' in available_providers:
    EXECUTION_PROVIDER = 'TensorrtExecutionProvider'
    print(f"\n✅ Usando: {EXECUTION_PROVIDER}")
else:
    EXECUTION_PROVIDER = 'CPUExecutionProvider'
    print(f"\n⚠️  No hay proveedor GPU disponible, usando: {EXECUTION_PROVIDER}")
    print("   Para habilitar GPU, instalar:")
    print("   - CUDA: pip install onnxruntime-gpu")
    print("   - DirectML: pip install onnxruntime-directml")

## 3. Definir Parámetros del Experimento de Monte Carlo

In [ ]:
# Parámetros del experimento de Monte Carlo
MONTE_CARLO_ITERATIONS = 100  # Número de experimentos
CONFIDENCE_LEVEL = 0.95     # Nivel de confianza para intervalos
MONITORING_INTERVAL = 0.1   # Intervalo de monitoreo GPU en segundos
SAMPLE_SIZE_PER_EXP = 3     # Número de imágenes por experimento

# Configuración de directorios
TEST_FOLDERS = [
    "Testig_Montecarlo/1", 
    "Testig_Montecarlo/2", 
    "Testig_Montecarlo/3", 
    "Testig_Montecarlo/4", 
    "Testig_Montecarlo/5"
]

# Configuración de modelos ONNX
MODEL_CONFIGS = {
    'yolo_path': "models/best_etiquetas.onnx",
    'ocr_path': 'models/model_ocr.onnx',
    'segrot_path': 'models/model_rotseg.onnx',
    'execution_provider': 'CUDAExecutionProvider'
}

# Estructura para almacenar resultados
results = []

print("🔧 Configuración del experimento:")
print(f"   • Iteraciones de Monte Carlo: {MONTE_CARLO_ITERATIONS}")
print(f"   • Nivel de confianza: {CONFIDENCE_LEVEL}")
print(f"   • Intervalo de monitoreo: {MONITORING_INTERVAL}s")
print(f"   • Imágenes por experimento: {SAMPLE_SIZE_PER_EXP}")
print(f"   • Carpetas de test disponibles: {len(TEST_FOLDERS)}")
print(f"   • Proveedor de ejecución: {EXECUTION_PROVIDER}")

## 4. Funciones de Monitoreo de GPU

In [ ]:
class GPUMonitor:
    """Clase para monitorear el uso de GPU durante la inferencia"""
    
    def __init__(self, interval=0.1, device_id=0):
        self.interval = interval
        self.device_id = device_id
        self.gpu_data = []
        self.stop_monitoring = False
        self.monitoring_thread = None
        
        if GPU_AVAILABLE:
            try:
                self.handle = pynvml.nvmlDeviceGetHandleByIndex(device_id)
                self.gpu_name = pynvml.nvmlDeviceGetName(self.handle)
                print(f"✅ Monitor GPU inicializado para: {self.gpu_name}")
            except Exception as e:
                print(f"❌ Error inicializando monitor GPU: {e}")
                self.handle = None
        else:
            self.handle = None
            print("⚠️  GPU monitoring no disponible")
        
    def _monitor_loop(self):
        """Loop de monitoreo de GPU en hilo separado"""
        if not self.handle:
            return
            
        while not self.stop_monitoring:
            try:
                timestamp = time.time()
                
                # Obtener utilización de GPU
                utilization = pynvml.nvmlDeviceGetUtilizationRates(self.handle)
                
                # Obtener información de memoria
                memory_info = pynvml.nvmlDeviceGetMemoryInfo(self.handle)
                
                # Obtener temperatura
                temperature = pynvml.nvmlDeviceGetTemperature(self.handle, pynvml.NVML_TEMPERATURE_GPU)
                
                # Obtener consumo de energía
                try:
                    power_usage = pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0  # mW to W
                except:
                    power_usage = 0
                
                self.gpu_data.append({
                    'timestamp': timestamp,
                    'gpu_utilization': utilization.gpu,
                    'memory_utilization': utilization.memory,
                    'memory_used_mb': memory_info.used / 1024**2,
                    'memory_total_mb': memory_info.total / 1024**2,
                    'memory_percent': (memory_info.used / memory_info.total) * 100,
                    'temperature': temperature,
                    'power_watts': power_usage
                })
                
                time.sleep(self.interval)
                
            except Exception as e:
                print(f"Error en monitoreo GPU: {e}")
                time.sleep(self.interval)
                
    def start_monitoring(self):
        """Iniciar monitoreo de GPU"""
        if not self.handle:
            print("⚠️  GPU monitoring no disponible, no se iniciará")
            return
            
        self.gpu_data = []
        self.stop_monitoring = False
        self.monitoring_thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.monitoring_thread.start()
        
    def stop_monitoring_and_get_data(self):
        """Detener monitoreo y obtener datos"""
        self.stop_monitoring = True
        if self.monitoring_thread:
            self.monitoring_thread.join(timeout=1.0)
        return self.gpu_data.copy()
    
    def get_statistics(self):
        """Calcular estadísticas del uso de GPU"""
        if not self.gpu_data:
            return None
            
        gpu_utils = np.array([data['gpu_utilization'] for data in self.gpu_data])
        mem_utils = np.array([data['memory_utilization'] for data in self.gpu_data])
        mem_used = np.array([data['memory_used_mb'] for data in self.gpu_data])
        mem_percent = np.array([data['memory_percent'] for data in self.gpu_data])
        temps = np.array([data['temperature'] for data in self.gpu_data])
        power = np.array([data['power_watts'] for data in self.gpu_data])
        
        stats = {
            'mean_gpu_util': np.mean(gpu_utils),
            'std_gpu_util': np.std(gpu_utils),
            'max_gpu_util': np.max(gpu_utils),
            'min_gpu_util': np.min(gpu_utils),
            'median_gpu_util': np.median(gpu_utils),
            'percentile_95_gpu': np.percentile(gpu_utils, 95),
            'percentile_99_gpu': np.percentile(gpu_utils, 99),
            
            'mean_mem_util': np.mean(mem_utils),
            'max_mem_util': np.max(mem_utils),
            'mean_mem_used_mb': np.mean(mem_used),
            'max_mem_used_mb': np.max(mem_used),
            'mean_mem_percent': np.mean(mem_percent),
            
            'mean_temperature': np.mean(temps),
            'max_temperature': np.max(temps),
            'min_temperature': np.min(temps),
            
            'mean_power_watts': np.mean(power),
            'max_power_watts': np.max(power),
            'min_power_watts': np.min(power),
            
            'gpu_util_series': gpu_utils,
            'mem_used_series': mem_used,
            'temperature_series': temps,
            'power_series': power,
            'num_samples': len(self.gpu_data)
        }
        
        return stats

# Función auxiliar para calcular intervalos de confianza
def calculate_confidence_interval(data, confidence=0.95):
    """Calcular intervalo de confianza para una muestra"""
    n = len(data)
    if n < 2:
        return (np.mean(data), np.mean(data))
    mean = np.mean(data)
    std_err = stats.sem(data)
    h = std_err * stats.t.ppf((1 + confidence) / 2., n-1)
    return mean - h, mean + h

print("🔧 Funciones de monitoreo de GPU configuradas")
if GPU_AVAILABLE:
    print(f"   • GPU disponible para monitoreo")
else:
    print(f"   ⚠️  Monitoreo GPU no disponible")

## 5. Pipeline de Procesamiento de Imágenes con GPU

In [ ]:
class ImageProcessingPipelineGPU:
    """Pipeline de procesamiento con modelos ONNX optimizado para GPU"""
    
    def __init__(self):
        self.config = MODEL_CONFIGS
        self.convert = TextConverter()
        self.models = {}
        
        # Normalización ImageNet
        self.MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        self.STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)
        
    def load_models(self):
        """Cargar modelos ONNX con proveedor de ejecución configurado"""
        try:
            # Configurar proveedor de ejecución
            providers = ['CUDAExecutionProvider']
            
            # YOLO con ultralytics
            self.models['yolo'] = YOLO(self.config['yolo_path'], task="obb")
            
            # Modelos ONNX con GPU
            self.models['ocr'] = ort.InferenceSession(
                self.config['ocr_path'], 
                providers=providers
            )
            self.models['segrot'] = ort.InferenceSession(
                self.config['segrot_path'], 
                providers=providers
            )
            
            # Verificar que los modelos estén usando el proveedor correcto
            print(f"✅ Modelos ONNX cargados exitosamente")
            print(f"   • OCR provider: {self.models['ocr'].get_providers()}")
            print(f"   • SegRot provider: {self.models['segrot'].get_providers()}")
            
            return True
            
        except Exception as e:
            print(f"❌ Error cargando modelos: {e}")
            return False
    
    def preprocess_image(self, image, size):
        """Preprocesamiento de imagen con normalización ImageNet"""
        image_resized = cv2.resize(image, size)
        image_processed = np.transpose(image_resized, (2, 0, 1))  # HWC to CHW
        image_processed = np.expand_dims(image_processed, axis=0)  # Add batch dimension
        image_processed = image_processed.astype(np.float32) / 255.0
        image_processed = (image_processed - self.MEAN) / self.STD
        return image_processed
    
    def rotate_image(self, image, angle):
        """Rotar imagen por un ángulo específico"""
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(image, M, (w, h))
        return rotated
    
    def seg_rot(self, image):
        """Segmentación y rotación de imagen"""
        w, h = image.shape[1], image.shape[0]
        img_trans = self.preprocess_image(image, (224, 224))
        
        # Inferencia ONNX
        results = self.models['segrot'].run(
            None, 
            {self.models['segrot'].get_inputs()[0].name: img_trans}
        )
        
        mask, angle = results
        angle = np.squeeze(angle)
        mask = np.squeeze(mask)
        mask = (mask > 0.5).astype(np.uint8) * 255
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        angle = np.argmax(angle) * 5
        
        # Rotar imagen y máscara
        img = self.rotate_image(image, -angle)
        predicted_mask = self.rotate_image(mask, -angle)
        
        # Extraer contorno principal
        contours, _ = cv2.findContours(predicted_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            cnt = max(contours, key=cv2.contourArea)
            rect = cv2.minAreaRect(cnt)
            box = cv2.boxPoints(rect)
            box = np.intp(box)
            
            src_pts = box.astype("float32")
            
            # Ordenar puntos consistentemente
            rect_pts = np.zeros((4, 2), dtype="float32")
            s = src_pts.sum(axis=1)
            rect_pts[0] = src_pts[np.argmin(s)]
            rect_pts[2] = src_pts[np.argmax(s)]
            
            diff = np.diff(src_pts, axis=1)
            rect_pts[1] = src_pts[np.argmin(diff)]
            rect_pts[3] = src_pts[np.argmax(diff)]
            
            # Calcular dimensiones
            W = int(np.linalg.norm(rect_pts[0] - rect_pts[1]))
            H = int(np.linalg.norm(rect_pts[1] - rect_pts[2]))
            
            if W > 0 and H > 0:
                dst_pts = np.array([[0, 0], [W-1, 0], [W-1, H-1], [0, H-1]], dtype="float32")
                M = cv2.getPerspectiveTransform(rect_pts, dst_pts)
                warped = cv2.warpPerspective(img, M, (W, H))
                return warped
        
        return None
    
    def order_points(self, pts):
        """Ordenar puntos del rectángulo"""
        rect = np.zeros((4, 2), dtype="float32")
        s = pts.sum(axis=1)
        rect[0] = pts[np.argmin(s)]
        rect[2] = pts[np.argmax(s)]
        diff = np.diff(pts, axis=1)
        rect[1] = pts[np.argmin(diff)]
        rect[3] = pts[np.argmax(diff)]
        return rect
    
    def detect_fields(self, image):
        """Detección de campos en imagen"""
        etiquetas = {0: "apellido", 1: "nombre", 2: "numero_cedula"}
        result = self.models['yolo'](image, verbose=False, conf=0.75)
        boxes = result[0].obb
        recortes = []
        
        for j, box in enumerate(boxes):
            cls_id = int(box.cls[0])
            if cls_id in etiquetas:
                pts = box.xyxyxyxy[0].cpu().numpy().reshape(4, 2)
                rect = self.order_points(pts)
                
                # Calcular dimensiones
                (tl, tr, br, bl) = rect
                widthA = np.linalg.norm(br - bl)
                widthB = np.linalg.norm(tr - tl)
                heightA = np.linalg.norm(tr - br)
                heightB = np.linalg.norm(tl - bl)
                
                maxWidth = int(max(widthA, widthB))
                maxHeight = int(max(heightA, heightB))
                
                # Forzar lado largo horizontal
                if maxHeight > maxWidth:
                    maxWidth, maxHeight = maxHeight, maxWidth
                    dst = np.array([[0, 0], [0, maxHeight - 1], [maxWidth - 1, maxHeight - 1], [maxWidth - 1, 0]], dtype="float32")
                else:
                    dst = np.array([[0, 0], [maxWidth - 1, 0], [maxWidth - 1, maxHeight - 1], [0, maxHeight - 1]], dtype="float32")
                
                # Transformación de perspectiva
                M = cv2.getPerspectiveTransform(rect, dst)
                warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
                
                recortes.append({
                    "array": warped,
                    "tipo": etiquetas[cls_id]
                })
        
        return recortes
    
    def run_ocr(self, image_crop):
        """Ejecutar OCR en un recorte de imagen"""
        img_ocr_proc = self.preprocess_image(image_crop, (224, 112))
        
        results_ocr = self.models['ocr'].run(
            None, 
            {self.models['ocr'].get_inputs()[0].name: img_ocr_proc}
        )
        
        results = results_ocr[0]
        text, confidence = self.convert.decode(results, softmax_probs=True)
        return text, confidence
    
    def process_image(self, image_path):
        """Procesar una imagen completa a través del pipeline"""
        try:
            # Cargar imagen
            image = cv2.imread(image_path)
            if image is None:
                return None
                
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Pipeline de procesamiento
            img_rec = self.seg_rot(image)
            if img_rec is None:
                return None
            
            # Detección de campos
            recortes = self.detect_fields(img_rec)
            if not recortes:
                return None
            
            # OCR en cada recorte
            ocr_results = []
            for rec in recortes:
                text, confidence = self.run_ocr(rec['array'])
                ocr_results.append({
                    'field': rec['tipo'],
                    'text': text,
                    'confidence': confidence
                })
            
            return {
                'image_path': image_path,
                'fields_detected': len(recortes),
                'ocr_results': ocr_results
            }
            
        except Exception as e:
            print(f"Error procesando {image_path}: {e}")
            return None

print("🔧 Pipeline de procesamiento GPU configurado")

## 6. Ejecutar Experimentos de Monte Carlo con GPU

In [ ]:
def get_random_images(folders, sample_size):
    """Obtener muestra aleatoria de imágenes de las carpetas especificadas"""
    all_images = []
    
    for folder in folders:
        if os.path.exists(folder):
            extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
            for ext in extensions:
                all_images.extend(glob(os.path.join(folder, ext)))
    
    if len(all_images) < sample_size:
        print(f"⚠️  Solo se encontraron {len(all_images)} imágenes, usando todas")
        return all_images
    
    return random.sample(all_images, sample_size)

def run_monte_carlo_experiment():
    """Ejecutar el experimento completo de Monte Carlo con GPU"""
    
    print("🚀 Iniciando experimento de Monte Carlo con GPU")
    print("=" * 60)
    
    # Inicializar pipeline
    pipeline = ImageProcessingPipelineGPU()
    
    if not pipeline.load_models():
        print("❌ No se pudieron cargar los modelos, abortando...")
        return
    
    # Ejecutar múltiples experimentos
    for exp_num in range(1, MONTE_CARLO_ITERATIONS + 1):
        print(f"\n🔬 Experimento {exp_num}/{MONTE_CARLO_ITERATIONS}")
        
        # Obtener muestra aleatoria de imágenes
        sample_images = get_random_images(TEST_FOLDERS, SAMPLE_SIZE_PER_EXP)
        
        if not sample_images:
            print("❌ No se encontraron imágenes para procesar")
            continue
        
        # Detectar folders usados
        folders_used = set()
        for img_path in sample_images:
            for test_folder in TEST_FOLDERS:
                if test_folder in img_path:
                    folders_used.add(test_folder)
                    break
        
        print(f"📁 Procesando {len(sample_images)} imágenes:")
        for img in sample_images:
            print(f"   • {os.path.basename(img)}")
        
        # Configurar monitor de GPU
        gpu_monitor = GPUMonitor(interval=MONITORING_INTERVAL)
        
        # Iniciar monitoreo y procesamiento
        start_time = time.time()
        gpu_monitor.start_monitoring()
        
        # Procesar cada imagen
        processing_results = []
        for img_path in sample_images:
            result = pipeline.process_image(img_path)
            if result:
                processing_results.append(result)
                print(f"✅ {os.path.basename(img_path)}: {result['fields_detected']} campos detectados")
            else:
                print(f"⚠️  {os.path.basename(img_path)}: Error en procesamiento")
        
        # Detener monitoreo
        end_time = time.time()
        gpu_data = gpu_monitor.stop_monitoring_and_get_data()
        gpu_stats = gpu_monitor.get_statistics()
        
        # Almacenar resultados del experimento
        experiment_result = {
            'experiment_number': exp_num,
            'start_time': start_time,
            'end_time': end_time,
            'duration': end_time - start_time,
            'images_processed': len(processing_results),
            'images_failed': len(sample_images) - len(processing_results),
            'sample_images': [os.path.basename(img) for img in sample_images],
            'sample_images_full_path': sample_images,
            'folders_used': list(folders_used),
            'gpu_stats': gpu_stats,
            'gpu_raw_data': gpu_data,
            'processing_results': processing_results,
            'execution_provider': 'CUDAExecutionProvider'
        }
        
        results.append(experiment_result)
        
        # Mostrar resumen del experimento
        if gpu_stats:
            print(f"⏱️  Duración: {experiment_result['duration']:.2f}s")
            print(f"📈 GPU utilización promedio: {gpu_stats['mean_gpu_util']:.1f}%")
            print(f"📊 GPU utilización máxima: {gpu_stats['max_gpu_util']:.1f}%")
            print(f"💾 Memoria GPU promedio: {gpu_stats['mean_mem_used_mb']:.1f} MB")
            print(f"💾 Memoria GPU máxima: {gpu_stats['max_mem_used_mb']:.1f} MB")
            print(f"🌡️  Temperatura promedio: {gpu_stats['mean_temperature']:.1f}°C")
            print(f"⚡ Potencia promedio: {gpu_stats['mean_power_watts']:.1f}W")
            print(f"📋 Muestras GPU: {gpu_stats['num_samples']}")
        
        # Pausa entre experimentos
        time.sleep(2)
    
    print("\n🎉 Experimentos de Monte Carlo completados")
    print("=" * 60)
    
    # Resumen general
    if results:
        print(f"\n📊 Resumen General:")
        print(f"   • Experimentos completados: {len(results)}")
        
        # Estadísticas agregadas
        durations = [exp['duration'] for exp in results]
        gpu_utils = [exp['gpu_stats']['mean_gpu_util'] for exp in results if exp['gpu_stats']]
        mem_used = [exp['gpu_stats']['mean_mem_used_mb'] for exp in results if exp['gpu_stats']]
        
        if durations:
            print(f"   • Duración promedio: {np.mean(durations):.2f}s ± {np.std(durations):.2f}s")
        if gpu_utils:
            print(f"   • GPU utilización promedio: {np.mean(gpu_utils):.1f}% ± {np.std(gpu_utils):.1f}%")
        if mem_used:
            print(f"   • Memoria GPU promedio: {np.mean(mem_used):.1f} MB ± {np.std(mem_used):.1f} MB")

# Ejecutar el experimento
run_monte_carlo_experiment()

## 7. Análisis Estadístico del Uso de GPU

In [ ]:
def analyze_experiment_statistics():
    """Realizar análisis estadístico detallado de los experimentos"""
    
    print("📊 ANÁLISIS ESTADÍSTICO DE EXPERIMENTOS GPU")
    print("=" * 60)
    
    if not results:
        print("⚠️  No hay datos para analizar")
        return None
    
    # Extraer métricas de todos los experimentos
    durations = []
    gpu_utils = []
    mem_used = []
    temperatures = []
    power_usage = []
    images_processed = []
    
    for exp in results:
        durations.append(exp['duration'])
        images_processed.append(exp['images_processed'])
        
        if exp['gpu_stats']:
            gpu_utils.append(exp['gpu_stats']['mean_gpu_util'])
            mem_used.append(exp['gpu_stats']['mean_mem_used_mb'])
            temperatures.append(exp['gpu_stats']['mean_temperature'])
            power_usage.append(exp['gpu_stats']['mean_power_watts'])
    
    # Calcular estadísticas descriptivas
    analysis = {}
    
    # Duración
    analysis['duration'] = {
        'mean': np.mean(durations),
        'std': np.std(durations),
        'min': np.min(durations),
        'max': np.max(durations),
        'median': np.median(durations),
        'ci_lower': calculate_confidence_interval(durations, CONFIDENCE_LEVEL)[0],
        'ci_upper': calculate_confidence_interval(durations, CONFIDENCE_LEVEL)[1]
    }
    
    # GPU Utilization
    if gpu_utils:
        analysis['gpu_utilization'] = {
            'mean': np.mean(gpu_utils),
            'std': np.std(gpu_utils),
            'min': np.min(gpu_utils),
            'max': np.max(gpu_utils),
            'median': np.median(gpu_utils),
            'ci_lower': calculate_confidence_interval(gpu_utils, CONFIDENCE_LEVEL)[0],
            'ci_upper': calculate_confidence_interval(gpu_utils, CONFIDENCE_LEVEL)[1]
        }
    
    # Memory Usage
    if mem_used:
        analysis['memory_usage_mb'] = {
            'mean': np.mean(mem_used),
            'std': np.std(mem_used),
            'min': np.min(mem_used),
            'max': np.max(mem_used),
            'median': np.median(mem_used),
            'ci_lower': calculate_confidence_interval(mem_used, CONFIDENCE_LEVEL)[0],
            'ci_upper': calculate_confidence_interval(mem_used, CONFIDENCE_LEVEL)[1]
        }
    
    # Temperature
    if temperatures:
        analysis['temperature'] = {
            'mean': np.mean(temperatures),
            'std': np.std(temperatures),
            'min': np.min(temperatures),
            'max': np.max(temperatures),
            'median': np.median(temperatures)
        }
    
    # Power Usage
    if power_usage:
        analysis['power_watts'] = {
            'mean': np.mean(power_usage),
            'std': np.std(power_usage),
            'min': np.min(power_usage),
            'max': np.max(power_usage),
            'median': np.median(power_usage)
        }
    
    # Imprimir resultados
    print("\n🔍 Estadísticas Descriptivas:")
    print("-" * 40)
    
    print(f"\n⏱️  Duración del procesamiento:")
    print(f"   • Media: {analysis['duration']['mean']:.3f}s ± {analysis['duration']['std']:.3f}s")
    print(f"   • IC 95%: [{analysis['duration']['ci_lower']:.3f}, {analysis['duration']['ci_upper']:.3f}]s")
    print(f"   • Rango: [{analysis['duration']['min']:.3f}, {analysis['duration']['max']:.3f}]s")
    
    if 'gpu_utilization' in analysis:
        print(f"\n📈 Utilización de GPU:")
        print(f"   • Media: {analysis['gpu_utilization']['mean']:.2f}% ± {analysis['gpu_utilization']['std']:.2f}%")
        print(f"   • IC 95%: [{analysis['gpu_utilization']['ci_lower']:.2f}, {analysis['gpu_utilization']['ci_upper']:.2f}]%")
        print(f"   • Rango: [{analysis['gpu_utilization']['min']:.2f}, {analysis['gpu_utilization']['max']:.2f}]%")
    
    if 'memory_usage_mb' in analysis:
        print(f"\n💾 Uso de Memoria GPU:")
        print(f"   • Media: {analysis['memory_usage_mb']['mean']:.2f} MB ± {analysis['memory_usage_mb']['std']:.2f} MB")
        print(f"   • IC 95%: [{analysis['memory_usage_mb']['ci_lower']:.2f}, {analysis['memory_usage_mb']['ci_upper']:.2f}] MB")
        print(f"   • Rango: [{analysis['memory_usage_mb']['min']:.2f}, {analysis['memory_usage_mb']['max']:.2f}] MB")
    
    if 'temperature' in analysis:
        print(f"\n🌡️  Temperatura GPU:")
        print(f"   • Media: {analysis['temperature']['mean']:.1f}°C ± {analysis['temperature']['std']:.1f}°C")
        print(f"   • Rango: [{analysis['temperature']['min']:.1f}, {analysis['temperature']['max']:.1f}]°C")
    
    if 'power_watts' in analysis:
        print(f"\n⚡ Consumo de Potencia:")
        print(f"   • Media: {analysis['power_watts']['mean']:.2f}W ± {analysis['power_watts']['std']:.2f}W")
        print(f"   • Rango: [{analysis['power_watts']['min']:.2f}, {analysis['power_watts']['max']:.2f}]W")
    
    # Throughput
    total_images = sum(images_processed)
    total_time = sum(durations)
    throughput = total_images / total_time if total_time > 0 else 0
    
    print(f"\n📊 Throughput:")
    print(f"   • Imágenes procesadas: {total_images}")
    print(f"   • Tiempo total: {total_time:.2f}s")
    print(f"   • Throughput: {throughput:.2f} imágenes/segundo")
    
    return analysis

# Ejecutar análisis
analysis_results = analyze_experiment_statistics()

## 8. Visualizaciones del Rendimiento de GPU

In [ ]:
def create_gpu_visualizations():
    """Crear visualizaciones comprehensivas del uso de GPU"""
    
    if not results:
        print("⚠️  No hay datos para visualizar")
        return
    
    print("📊 Generando visualizaciones...")
    
    # Configurar figura con múltiples subplots
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Distribución de utilización de GPU
    ax1 = fig.add_subplot(gs[0, 0])
    gpu_utils = [exp['gpu_stats']['mean_gpu_util'] for exp in results if exp['gpu_stats']]
    if gpu_utils:
        ax1.hist(gpu_utils, bins=20, color='#4CAF50', alpha=0.7, edgecolor='black')
        ax1.axvline(np.mean(gpu_utils), color='red', linestyle='--', linewidth=2, label=f'Media: {np.mean(gpu_utils):.1f}%')
        ax1.set_xlabel('Utilización GPU (%)', fontsize=10)
        ax1.set_ylabel('Frecuencia', fontsize=10)
        ax1.set_title('Distribución de Utilización de GPU', fontsize=12, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
    # 2. Distribución de uso de memoria
    ax2 = fig.add_subplot(gs[0, 1])
    mem_used = [exp['gpu_stats']['mean_mem_used_mb'] for exp in results if exp['gpu_stats']]
    if mem_used:
        ax2.hist(mem_used, bins=20, color='#2196F3', alpha=0.7, edgecolor='black')
        ax2.axvline(np.mean(mem_used), color='red', linestyle='--', linewidth=2, label=f'Media: {np.mean(mem_used):.1f} MB')
        ax2.set_xlabel('Memoria GPU (MB)', fontsize=10)
        ax2.set_ylabel('Frecuencia', fontsize=10)
        ax2.set_title('Distribución de Uso de Memoria GPU', fontsize=12, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    # 3. Distribución de temperatura
    ax3 = fig.add_subplot(gs[0, 2])
    temps = [exp['gpu_stats']['mean_temperature'] for exp in results if exp['gpu_stats']]
    if temps:
        ax3.hist(temps, bins=20, color='#FF9800', alpha=0.7, edgecolor='black')
        ax3.axvline(np.mean(temps), color='red', linestyle='--', linewidth=2, label=f'Media: {np.mean(temps):.1f}°C')
        ax3.set_xlabel('Temperatura (°C)', fontsize=10)
        ax3.set_ylabel('Frecuencia', fontsize=10)
        ax3.set_title('Distribución de Temperatura GPU', fontsize=12, fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    
    # 4. Serie temporal de utilización GPU (primer experimento)
    ax4 = fig.add_subplot(gs[1, :])
    if results and results[0]['gpu_stats']:
        gpu_series = results[0]['gpu_stats']['gpu_util_series']
        time_points = np.arange(len(gpu_series)) * MONITORING_INTERVAL
        ax4.plot(time_points, gpu_series, color='#4CAF50', linewidth=2, label='Utilización GPU')
        ax4.fill_between(time_points, gpu_series, alpha=0.3, color='#4CAF50')
        ax4.set_xlabel('Tiempo (s)', fontsize=10)
        ax4.set_ylabel('Utilización GPU (%)', fontsize=10)
        ax4.set_title('Serie Temporal de Utilización GPU (Experimento 1)', fontsize=12, fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
    # 5. Box plot de métricas
    ax5 = fig.add_subplot(gs[2, 0])
    gpu_utils = [exp['gpu_stats']['mean_gpu_util'] for exp in results if exp['gpu_stats']]
    if gpu_utils:
        bp = ax5.boxplot([gpu_utils], labels=['GPU Utilization'], patch_artist=True)
        bp['boxes'][0].set_facecolor('#4CAF50')
        ax5.set_ylabel('Utilización (%)', fontsize=10)
        ax5.set_title('Box Plot: Utilización GPU', fontsize=12, fontweight='bold')
        ax5.grid(True, alpha=0.3, axis='y')
    
    # 6. Scatter plot: Duración vs GPU utilización
    ax6 = fig.add_subplot(gs[2, 1])
    durations = [exp['duration'] for exp in results]
    gpu_utils = [exp['gpu_stats']['mean_gpu_util'] for exp in results if exp['gpu_stats']]
    if durations and gpu_utils and len(durations) == len(gpu_utils):
        ax6.scatter(durations, gpu_utils, alpha=0.6, s=100, color='#4CAF50', edgecolors='black')
        ax6.set_xlabel('Duración (s)', fontsize=10)
        ax6.set_ylabel('Utilización GPU (%)', fontsize=10)
        ax6.set_title('Duración vs Utilización GPU', fontsize=12, fontweight='bold')
        ax6.grid(True, alpha=0.3)
        
        # Correlación
        if len(durations) > 1:
            corr = np.corrcoef(durations, gpu_utils)[0, 1]
            ax6.text(0.05, 0.95, f'Correlación: {corr:.3f}', 
                    transform=ax6.transAxes, fontsize=10, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 7. Consumo de potencia promedio
    ax7 = fig.add_subplot(gs[2, 2])
    power = [exp['gpu_stats']['mean_power_watts'] for exp in results if exp['gpu_stats']]
    if power:
        ax7.hist(power, bins=20, color='#F44336', alpha=0.7, edgecolor='black')
        ax7.axvline(np.mean(power), color='darkred', linestyle='--', linewidth=2, label=f'Media: {np.mean(power):.1f}W')
        ax7.set_xlabel('Potencia (W)', fontsize=10)
        ax7.set_ylabel('Frecuencia', fontsize=10)
        ax7.set_title('Distribución de Consumo de Potencia', fontsize=12, fontweight='bold')
        ax7.legend()
        ax7.grid(True, alpha=0.3)
    
    plt.suptitle(f'Análisis de Rendimiento GPU - ONNX ({MONTE_CARLO_ITERATIONS} Experimentos)', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    plt.savefig('gpu_performance_analysis.png', dpi=300, bbox_inches='tight')
    print("✅ Visualizaciones guardadas en 'gpu_performance_analysis.png'")
    plt.show()

# Generar visualizaciones
create_gpu_visualizations()

## 9. Exportar Resultados

In [ ]:
def export_results_to_csv():
    """Exportar resultados a archivo CSV"""
    
    if not results:
        print("⚠️  No hay datos para exportar")
        return
    
    # Crear DataFrame con métricas principales
    data = []
    
    for exp in results:
        row = {
            'experiment_number': exp['experiment_number'],
            'duration_seconds': exp['duration'],
            'images_processed': exp['images_processed'],
            'images_failed': exp['images_failed'],
            'execution_provider': exp['execution_provider']
        }
        
        if exp['gpu_stats']:
            row.update({
                'gpu_util_mean': exp['gpu_stats']['mean_gpu_util'],
                'gpu_util_max': exp['gpu_stats']['max_gpu_util'],
                'gpu_util_std': exp['gpu_stats']['std_gpu_util'],
                'mem_used_mb_mean': exp['gpu_stats']['mean_mem_used_mb'],
                'mem_used_mb_max': exp['gpu_stats']['max_mem_used_mb'],
                'mem_percent_mean': exp['gpu_stats']['mean_mem_percent'],
                'temperature_mean': exp['gpu_stats']['mean_temperature'],
                'temperature_max': exp['gpu_stats']['max_temperature'],
                'power_watts_mean': exp['gpu_stats']['mean_power_watts'],
                'power_watts_max': exp['gpu_stats']['max_power_watts'],
                'num_samples': exp['gpu_stats']['num_samples']
            })
        
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Guardar CSV
    filename = 'monte_carlo_gpu_results.csv'
    df.to_csv(filename, index=False)
    
    print(f"✅ Resultados exportados a '{filename}'")
    print(f"📊 Total de experimentos: {len(df)}")
    print("\n📋 Resumen de columnas:")
    print(df.describe().T)
    
    return df

# Exportar resultados
results_df = export_results_to_csv()

## 10. Resumen Final

In [ ]:
print("=" * 60)
print("🎯 RESUMEN FINAL DEL EXPERIMENTO")
print("=" * 60)

if results:
    print(f"\n✅ Experimentos completados: {len(results)}/{MONTE_CARLO_ITERATIONS}")
    print(f"🔧 Proveedor de ejecución: {MODEL_CONFIGS['execution_provider']}")
    
    # Estadísticas generales
    total_images = sum([exp['images_processed'] for exp in results])
    total_time = sum([exp['duration'] for exp in results])
    throughput = total_images / total_time if total_time > 0 else 0
    
    print(f"\n📊 Métricas Generales:")
    print(f"   • Total de imágenes procesadas: {total_images}")
    print(f"   • Tiempo total de procesamiento: {total_time:.2f}s")
    print(f"   • Throughput promedio: {throughput:.2f} imágenes/segundo")
    
    # Métricas de GPU
    gpu_utils = [exp['gpu_stats']['mean_gpu_util'] for exp in results if exp['gpu_stats']]
    mem_used = [exp['gpu_stats']['mean_mem_used_mb'] for exp in results if exp['gpu_stats']]
    temps = [exp['gpu_stats']['mean_temperature'] for exp in results if exp['gpu_stats']]
    power = [exp['gpu_stats']['mean_power_watts'] for exp in results if exp['gpu_stats']]
    
    if gpu_utils:
        print(f"\n📈 Utilización de GPU:")
        print(f"   • Promedio: {np.mean(gpu_utils):.2f}% ± {np.std(gpu_utils):.2f}%")
        print(f"   • Rango: [{np.min(gpu_utils):.2f}%, {np.max(gpu_utils):.2f}%]")
    
    if mem_used:
        print(f"\n💾 Memoria GPU:")
        print(f"   • Promedio: {np.mean(mem_used):.2f} MB ± {np.std(mem_used):.2f} MB")
        print(f"   • Máximo: {np.max(mem_used):.2f} MB")
    
    if temps:
        print(f"\n🌡️  Temperatura:")
        print(f"   • Promedio: {np.mean(temps):.1f}°C ± {np.std(temps):.1f}°C")
        print(f"   • Máximo: {np.max(temps):.1f}°C")
    
    if power:
        print(f"\n⚡ Consumo de Potencia:")
        print(f"   • Promedio: {np.mean(power):.2f}W ± {np.std(power):.2f}W")
        print(f"   • Máximo: {np.max(power):.2f}W")
    
    print("\n📁 Archivos generados:")
    print("   • gpu_performance_analysis.png")
    print("   • monte_carlo_gpu_results.csv")
    
else:
    print("\n⚠️  No se completó ningún experimento")

print("\n" + "=" * 60)
print("✨ Experimento finalizado")
print("=" * 60)